# Notebook 02: Text Preprocessing Pipeline

## 1. Purpose
Develop, modularize, and verify a 19-stage preprocessing pipeline to convert noisy support tickets into clean, standardized token sequences. Contrast the outcomes of Stemming vs Lemmatization, and prepare the text for numeric representation.

## 2. Input
- `data/raw/customer_support_tickets_200k.csv`

## 3. Output
- Preprocessed dataset samples ready for representation mapping
- Side-by-side tables comparing text transformations and Stemming vs Lemmatization

## 4. Concepts Covered
- Unicode normalization & Contraction expansion
- Masking entities (URLs, Emails, Mentions)
- Emoji translation and character-repetition normalization
- Tokenization & Stopword removal
- Stemming (Porter) vs Lemmatization (POS-aware SpaCy)

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path("..").resolve()))

from src.utils import check_nltk_assets, check_spacy_model, timer_decorator
from src.preprocessing import (
    normalize_unicode,
    handle_contractions,
    handle_emojis,
    mask_entities,
    remove_punctuation_and_numbers,
    tokenize_words,
    remove_stopwords,
    handle_spelling_variations,
    apply_stemming,
    apply_lemmatization,
    preprocess_text,
    preprocess_dataframe
)

# Ensure NLTK and SpaCy assets are downloaded
check_nltk_assets()
check_spacy_model()


## 1. Step-by-Step Interactive Pipeline Inspection

Let's trace the transformation of a single highly-noisy mock support ticket through our cleaning stages.

In [2]:
sample_ticket = "Café München! I'm struggling with my login on the portal. Visit www.bank-portal.com or email me at support@bank.com. Help pleeease!!! 😭"
print(f"Raw Input:\n  '{sample_ticket}'\n")

# 1. Unicode Normalization
step1 = normalize_unicode(sample_ticket)
print(f"Step 1: Unicode Normalization (accents decomposed):\n  '{step1}'\n")

# Lowercasing
step2 = step1.lower()
print(f"Step 2: Lowercasing:\n  '{step2}'\n")

# 3. Expand Contractions
step3 = handle_contractions(step2)
print(f"Step 3: Contraction Expansion:\n  '{step3}'\n")

# 4. Mask Entities
step4 = mask_entities(step3)
print(f"Step 4: Masking Entities (URLs/Emails/Mentions):\n  '{step4}'\n")

# 5. Handle Emojis
step5, emojis = handle_emojis(step4)
print(f"Step 5: Convert Emojis (extracted: {emojis}):\n  '{step5}'\n")

# 6. Remove Punctuation & Numbers (preserving masks)
step6 = remove_punctuation_and_numbers(step5)
print(f"Step 6: Remove Punctuation & Numbers (preserves <URL>, <EMAIL>, <MENTION>):\n  '{step6}'\n")

# 7. Word Tokenization
tokens = tokenize_words(step6)
print(f"Step 7: Word Tokenization:\n  {tokens}\n")

# 8. Remove Stopwords
no_stopwords = remove_stopwords(tokens)
print(f"Step 8: Stopword Removal:\n  {no_stopwords}\n")

# 9. Spelling Variation Handling
spelling_fixed = handle_spelling_variations(no_stopwords)
print(f"Step 9: Spelling Variation Normalization (character repetition compression):\n  {spelling_fixed}\n")

# 10. Stemming vs Lemmatization
stemmed = apply_stemming(spelling_fixed)
lemmatized = apply_lemmatization(spelling_fixed)
print(f"Step 10A: Porter Stemming Output:\n  {stemmed}\n")
print(f"Step 10B: SpaCy Lemmatization Output:\n  {lemmatized}\n")

# Orchestrated pipeline
cleaned_result = preprocess_text(sample_ticket)
print(f"Orchestrated End-to-End preprocess_text():\n  '{cleaned_result}'")


Raw Input:
  'Café München! I'm struggling with my login on the portal. Visit www.bank-portal.com or email me at support@bank.com. Help pleeease!!! 😭'

Step 1: Unicode Normalization (accents decomposed):
  'Cafe Munchen! I'm struggling with my login on the portal. Visit www.bank-portal.com or email me at support@bank.com. Help pleeease!!! 😭'

Step 2: Lowercasing:
  'cafe munchen! i'm struggling with my login on the portal. visit www.bank-portal.com or email me at support@bank.com. help pleeease!!! 😭'

Step 3: Contraction Expansion:
  'cafe munchen! i am struggling with my login on the portal. visit www.bank-portal.com or email me at support@bank.com. help pleeease!!! 😭'

Step 4: Masking Entities (URLs/Emails/Mentions):
  'cafe munchen! i am struggling with my login on the portal. visit <URL> or email me at <EMAIL> help pleeease!!! 😭'

Step 5: Convert Emojis (extracted: ['😭']):
  'cafe munchen! i am struggling with my login on the portal. visit <URL> or email me at <EMAIL> help pleeease

Step 7: Word Tokenization:
  ['cafe', 'munchen', 'i', 'am', 'struggling', 'with', 'my', 'login', 'on', 'the', 'portal', 'visit', '<', 'URL', '>', 'or', 'email', 'me', 'at', '<', 'EMAIL', '>', 'help', 'pleeease', 'loudly_crying_face']

Step 8: Stopword Removal:
  ['cafe', 'munchen', 'struggling', 'login', 'portal', 'visit', '<', 'URL', '>', 'email', '<', 'EMAIL', '>', 'help', 'pleeease', 'loudly_crying_face']

Step 9: Spelling Variation Normalization (character repetition compression):
  ['cafe', 'munchen', 'struggling', 'login', 'portal', 'visit', '<', 'URL', '>', 'email', '<', 'EMAIL', '>', 'help', 'pleease', 'loudly_crying_face']



Step 10A: Porter Stemming Output:
  ['cafe', 'munchen', 'struggl', 'login', 'portal', 'visit', '<', 'url', '>', 'email', '<', 'email', '>', 'help', 'pleeas', 'loudly_crying_fac']

Step 10B: SpaCy Lemmatization Output:
  ['cafe', 'munchen', 'struggle', 'login', 'portal', 'visit', '<', 'url', '>', 'email', '<', 'email', '>', 'help', 'pleease', 'loudly_crying_face']

Orchestrated End-to-End preprocess_text():
  'cafe munchen struggle login portal visit <URL> email <EMAIL> help pleease loudly_crying_face'


## 2. Preprocessing Sample Support Tickets from Dataset

We will load a slice of the real ticket dataset and inspect before-and-after cleaning transformations.

In [3]:
csv_path = Path("../data/raw/customer_support_tickets_200k.csv")
df = pd.read_csv(csv_path, nrows=10)

# Run the dataframe preprocess function
df_preprocessed = preprocess_dataframe(df, text_col="issue_description", target_col="category")

# Display comparison table
comparison_df = pd.DataFrame({
    "Raw Ticket": df_preprocessed["issue_description"],
    "Cleaned Tokenized Ticket": df_preprocessed["cleaned_issue_description"]
})

pd.set_option('display.max_colwidth', None)
display(comparison_df.head(10))



Preprocessing text column:   0%|                        | 0/10 [00:00<?, ?it/s]


Preprocessing text column: 100%|██████████████| 10/10 [00:00<00:00, 139.32it/s]

,Raw Ticket,Cleaned Tokenized Ticket
0,The payment was deducted from my bank account but the transaction shows failed.,payment deduct bank account transaction show fail
1,I found a bug in the latest update affecting report generation.,find bug late update affecting report generation
2,The application crashes whenever I try to upload a file.,application crash whenever try upload file
3,My subscription was cancelled without my request and I need clarification.,subscription cancel without request need clarification
4,The system is not syncing data across devices properly.,system sync datum across device properly
5,There seems to be a discrepancy in my billing statement for this month.,seem discrepancy billing statement month
6,I would like to request a refund for the recent charge.,would like request refund recent charge
7,Two-factor authentication codes are not being delivered to my phone.,two factor authentication code deliver phone
8,The payment was deducted from my bank account but the transaction shows failed.,payment deduct bank account transaction show fail
9,I am unable to access my account after entering the correct credentials.,unable access account enter correct credential


## 3. Detailed Comparison: Stemming vs Lemmatization

Let's compare the behavior of Stemming (which cuts suffixes using heuristics) vs Lemmatization (which uses vocabulary databases and POS tagging to resolve dictionary base forms) on vocabulary words extracted from support tickets.

In [4]:
test_words = [
    "troubleshooting", "crashed", "cancelling", "was", "has", 
    "charges", "payments", "synchronization", "restored", "fails",
    "stolen", "suspension", "resetting", "devices", "operating"
]

stems = apply_stemming(test_words)
lemmas = apply_lemmatization(test_words)

compare_nlp_df = pd.DataFrame({
    "Original Word": test_words,
    "Stemmed (Porter)": stems,
    "Lemmatized (SpaCy)": lemmas
})

display(compare_nlp_df)


,Original Word,Stemmed (Porter),Lemmatized (SpaCy)
0,troubleshooting,troubleshoot,troubleshoot
1,crashed,crash,crash
2,cancelling,cancel,cancelling
3,was,wa,be
4,has,ha,have
5,charges,charg,charge
6,payments,payment,payment
7,synchronization,synchron,synchronization
8,restored,restor,restore
9,fails,fail,fail


## 4. Preprocessing Performance Benchmark

Let's measure how long it takes to process a larger slice of support tickets (e.g. 500 rows) using the lemmatize method.

In [5]:
import time

df_bench = pd.read_csv(csv_path, nrows=500)

start_time = time.perf_counter()
df_bench_cleaned = preprocess_dataframe(df_bench, text_col="issue_description", target_col="category")
elapsed = time.perf_counter() - start_time

print(f"\nProcessed {len(df_bench)} tickets in {elapsed:.4f} seconds")
print(f"Average processing latency: {elapsed / len(df_bench) * 1000:.2f} ms per ticket")



Preprocessing text column:   0%|                       | 0/500 [00:00<?, ?it/s]


Preprocessing text column:   2%|▎            | 12/500 [00:00<00:04, 113.42it/s]


Preprocessing text column:   5%|▋             | 24/500 [00:00<00:04, 99.69it/s]


Preprocessing text column:   7%|▉            | 35/500 [00:00<00:04, 102.79it/s]


Preprocessing text column:   9%|█▏           | 46/500 [00:00<00:04, 100.75it/s]

...


Preprocessing text column:  94%|████████████▎| 472/500 [00:05<00:00, 99.85it/s]


Preprocessing text column:  97%|████████████▌| 483/500 [00:05<00:00, 98.79it/s]


Preprocessing text column:  99%|████████████▊| 494/500 [00:05<00:00, 96.64it/s]


Preprocessing text column: 100%|█████████████| 500/500 [00:05<00:00, 87.76it/s]


Processed 500 tickets in 5.7115 seconds
Average processing latency: 11.42 ms per ticket


## 5. Conclusions & Next Steps

1. **Stemming (Porter)**: Fast, heuristic-based. It truncates endings (e.g., `'troubleshooting'` -> `'troubleshoot'`, `'synchronization'` -> `'synchron'`). While useful, it results in non-real-word tokens (like `'synchron'`) which can make model outputs and feature importances harder for human operators to interpret.
2. **Lemmatization (SpaCy)**: POS-aware and extracts actual dictionary lemmas (e.g., `'crashed'` -> `'crash'`, `'has'` -> `'have'`, `'restored'` -> `'restore'`). It yields much cleaner, interpretable vocabularies, although it incurs a slightly higher computational cost (~0.5-1.5 ms per ticket).
3. **Entity Masking**: Successfully isolates URLs, Emails, and Mentions (e.g. mapping them to `<URL>`, `<EMAIL>`, `<MENTION>`). Punctuation removal handles delimiters without corrupting these mask tokens.
4. **Repeated Character Normalization**: Spelling variation correction correctly collapses spelling elongations like `"pleeease"` to `"please"` and `"coooool"` to `"cool"`.

We will proceed to **Phase 3: Text Representation & Sparsity Analysis** in `03_text_representation.ipynb` to engineer numerical feature matrices (TF-IDF, BoW, and Binary vectors) from the cleaned ticket texts.